# rPPG - Deep HR Training (Colab / Kaggle)

Trains our models (ProPOSNet / PhysNet / TS-CAN) to >90% within +/-5 bpm on UBFC with
subject-independent and cross-dataset ('uneven data') validation.

**Set Runtime -> Change runtime type -> GPU (T4) before running.**

In [ ]:
# 1. Setup: clone repo + install
!git clone https://github.com/Zero-State-Logic/rPPG.git
%cd rPPG
!pip install -q -e .
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 2. Download UBFC-rPPG DATASET_2 (cloud IP is not Drive-quota-limited like the local box)
import gdown
gdown.download_folder('https://drive.google.com/drive/folders/1q4vWuF2GJvKP5xyeX8dxaJ2fmq97-4ai',
                      output='data/UBFC', quiet=False, remaining_ok=True)
!ls data/UBFC | head

In [ ]:
# 3. Train ProPOSNet (our novel model) - subject-independent split
!python scripts/train_deep.py --dataset ubfc --data-root data/UBFC --model proposnet \
    --eval-mode subject --epochs 30 --batch-size 4

In [ ]:
# 4. Baselines for the paper table
!python scripts/train_deep.py --dataset ubfc --data-root data/UBFC --model physnet --eval-mode subject --epochs 30
!python scripts/train_deep.py --dataset ubfc --data-root data/UBFC --model tscan   --eval-mode subject --epochs 30

## Cross-dataset ('uneven data') validation

Once PURE is available, train on UBFC and test on the *different* PURE distribution:

```
!python scripts/train_deep.py --dataset ubfc --data-root data/UBFC --model proposnet \
    --eval-mode cross --cross-dataset pure --cross-root data/PURE
```

## Kaggle (switch here when Colab's daily GPU limit hits)

Same cells. Settings -> Accelerator -> GPU T4 x2. Save `data/` as a private Kaggle Dataset to
skip re-downloading. Checkpoints are written to `artifacts/models/`.